# **Day 51: Dataclasses**

Many classes only **store data**. For such classes, writing `__init__`, `__repr__` and `__eq__` again and again is boring.

A **dataclass** writes these methods **for us automatically**. We just list the attributes with their types.

In [1]:
# Without a dataclass: a lot of repeated code
class BookOld:
    def __init__(self, title, author, price):
        self.title = title
        self.author = author
        self.price = price
    def __repr__(self):
        return f"BookOld(title={self.title!r}, author={self.author!r}, price={self.price})"
    def __eq__(self, other):
        return (self.title, self.author, self.price) == (other.title, other.author, other.price)

In [2]:
# With a dataclass: short and clean
from dataclasses import dataclass, field, asdict

@dataclass
class Book:
    title: str
    author: str
    price: float

b1 = Book("Pather Panchali", "Bibhutibhushan Bandyopadhyay", 250)
b2 = Book("Pather Panchali", "Bibhutibhushan Bandyopadhyay", 250)

print(b1)            # __repr__ is made automatically
print(b1 == b2)      # __eq__ is made automatically

Book(title='Pather Panchali', author='Bibhutibhushan Bandyopadhyay', price=250)
True


- `title: str` is a **type hint**. It tells the reader the expected type. Python does not check it, but it is required in a dataclass.

## **Default Values**
**Important:** For a list or dictionary as a default, we must use `field(default_factory=list)`. A normal `= []` is not allowed (remember the shared list problem from Day 22).

In [3]:
@dataclass
class Student:
    name: str
    roll: int
    school: str = "Hindu School"
    subjects: list = field(default_factory=list)

s = Student("Arka", 12)
s.subjects.append("Physics")
print(s)
print(Student("Bonny", 13))

Student(name='Arka', roll=12, school='Hindu School', subjects=['Physics'])
Student(name='Bonny', roll=13, school='Hindu School', subjects=[])


## **__post_init__: Extra Work After Creation**
`__post_init__` runs right after the automatic `__init__`. We use it for **checking values** or **calculating** other fields.

In [4]:
@dataclass
class Plot:
    mouza: str
    length_ft: float
    width_ft: float
    area_sqft: float = field(init=False)      # Not given by the user, calculated
    area_katha: float = field(init=False)

    def __post_init__(self):
        if self.length_ft <= 0 or self.width_ft <= 0:
            raise ValueError("Length and width must be positive")
        self.area_sqft = self.length_ft * self.width_ft
        self.area_katha = round(self.area_sqft / 720, 2)     # 1 katha = 720 sq ft in West Bengal

print(Plot("Keshpur", 40, 54))

Plot(mouza='Keshpur', length_ft=40, width_ft=54, area_sqft=2160, area_katha=3.0)


## **Example from Remote Sensing: A Scene Dataclass**
A satellite scene has a lot of information. A dataclass is perfect for storing it.

In [5]:
@dataclass
class Scene:
    scene_id: str
    satellite: str
    date: str
    cloud_cover: float
    path_row: str = ""
    bands: list = field(default_factory=list)

    def __post_init__(self):
        if not 0 <= self.cloud_cover <= 100:
            raise ValueError("Cloud cover must be between 0 and 100")

    @property
    def is_clear(self):
        return self.cloud_cover < 10

s1 = Scene("LC08_138044_20231110", "Landsat 8", "2023-11-10", 2.35, "138/044", ["B2", "B3", "B4", "B5"])
s2 = Scene("S2A_45QXF_20240705", "Sentinel-2", "2024-07-05", 76.0)

print(s1)
print(s1.is_clear, s2.is_clear)
print(asdict(s1))       # Convert to a dictionary (useful for saving as JSON)

Scene(scene_id='LC08_138044_20231110', satellite='Landsat 8', date='2023-11-10', cloud_cover=2.35, path_row='138/044', bands=['B2', 'B3', 'B4', 'B5'])
True False
{'scene_id': 'LC08_138044_20231110', 'satellite': 'Landsat 8', 'date': '2023-11-10', 'cloud_cover': 2.35, 'path_row': '138/044', 'bands': ['B2', 'B3', 'B4', 'B5']}


## **frozen=True: An Object That Cannot Be Changed**

In [6]:
@dataclass(frozen=True)
class Coord:
    lat: float
    lon: float

kolkata = Coord(22.5726, 88.3639)
print(kolkata)
# kolkata.lat = 0          # This will raise a FrozenInstanceError

# Frozen objects can be used as dictionary keys
names = {kolkata: "Kolkata"}
print(names[Coord(22.5726, 88.3639)])

Coord(lat=22.5726, lon=88.3639)
Kolkata


## **order=True: Objects That Can Be Sorted**
The objects are compared field by field, in the order the fields are written.

In [7]:
@dataclass(order=True)
class Match:
    points: int
    goal_difference: int
    team: str = field(compare=False)       # Do not use the name for comparing

table = [Match(22, 8, "East Bengal"), Match(25, 12, "Mohun Bagan"),
         Match(22, 11, "Mohammedan"), Match(18, -2, "Bhawanipore")]

for rank, m in enumerate(sorted(table, reverse=True), 1):
    print(rank, m.team, m.points, m.goal_difference)

1 Mohun Bagan 25 12
2 Mohammedan 22 11
3 East Bengal 22 8
4 Bhawanipore 18 -2


(The league table is made up.)

## **Practice Problem: Stock of Handicrafts**

In [8]:
@dataclass
class Item:
    name: str
    qty: int
    unit_price: float

    @property
    def value(self):
        return self.qty * self.unit_price

stock = [Item("Jute bag", 120, 45), Item("Sital pati", 15, 850),
         Item("Dokra horse", 8, 1200), Item("Baluchari saree", 3, 7500)]

for it in sorted(stock, key=lambda i: i.value, reverse=True):
    print(f"{it.name:<16}{it.qty:>4} x {it.unit_price:>7} = {it.value:>9,.0f}")
print(f"Total stock value: {sum(i.value for i in stock):,.0f}")

Baluchari saree    3 x    7500 =    22,500
Sital pati        15 x     850 =    12,750
Dokra horse        8 x    1200 =     9,600
Jute bag         120 x      45 =     5,400
Total stock value: 50,250


In [9]:
# Quick question
# Guess the output of this code cell
# from dataclasses import dataclass
# @dataclass
# class P:
#     x: int
#     y: int = 0
# print(P(3), P(3) == P(3, 0))